In [ ]:
# ! pip install -qU keras-rs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.6/94.6 kB 3.2 MB/s eta 0:00:00


### Claude explanation (not from the professor)
Selects **JAX** as the Keras backend. This must be set before `import keras`. TensorFlow is used only for `tf.data`.

In [1]:
import os
os.environ["KERAS_BACKEND"] = "jax"

import keras_rs
import keras
import tensorflow as tf
import sqlite3
import pandas as pd

# Datos

### Claude explanation (not from the professor)
Loads every `(reader, book, rating)` interaction.

In [2]:
conn = sqlite3.connect('datos/datos.db')

df_ratings = pd.read_sql("SELECT id_lector, id_libro, rating FROM interacciones", conn)
df_ratings.head()

,id_lector,id_libro,rating
0,bioaqua,las-puertas-de-anubis,4
1,bioaqua,antonio-b-el-ruso-ciudadano-de-tercera,10
2,bioaqua,la-momia,10
3,bioaqua,el-clan-del-oso-cavernario-los-hijos-de-la-tie...,10
4,bioaqua,dime-quien-soy-1,10


### Claude explanation (not from the professor)
Builds dictionaries that translate string IDs to consecutive integers and back. Embedding layers need integer indices.

In [3]:
unique_user_id = df_ratings["id_lector"].unique().tolist()
user_id_2_id_mapping = {t: i for i, t in enumerate(unique_user_id)}
id_2_user_user_mapping = {user_id_2_id_mapping[k]: k for k in user_id_2_id_mapping}

unique_item_id = df_ratings["id_libro"].unique().tolist()
item_id_2_id_mapping = {t: i for i, t in enumerate(unique_item_id)}
id_2_item_id_mapping = {item_id_2_id_mapping[k]: k for k in item_id_2_id_mapping}

### Claude explanation (not from the professor)
Replaces the string IDs in the DataFrame with their integer index.

In [4]:
df_ratings["id_lector"] = df_ratings["id_lector"].apply(lambda x: user_id_2_id_mapping[x])
df_ratings["id_libro"] = df_ratings["id_libro"].apply(lambda x: item_id_2_id_mapping[x])

df_ratings.head()

,id_lector,id_libro,rating
0,0,0,4
1,0,1,10
2,0,2,10
3,0,3,10
4,0,4,10


# Recuperación

### Claude explanation (not from the professor)
Dataset for the **retrieval** stage. The input is only the reader index; the target holds the book index and the rating rescaled to 0–1. Batches of 1024.

**Note:** the rating is divided by 10 here but by 9 in the ranking stage, so here it spans 0–0.9. **Note:** `skip(df_ratings.shape[0] - cut)` skips only 20% of the rows, so the test set is the *last 80%* and mostly overlaps training. It should be `skip(cut)`.

In [5]:
dataset_inputs = df_ratings['id_lector'].to_numpy(dtype="float32")
dataset_outputs = {
    'item_id': df_ratings['id_libro'].to_numpy(dtype="float32"),
    'rating': (df_ratings['rating'].to_numpy() - 1.0) / 10.0
}

dataset = tf.data.Dataset.from_tensor_slices((dataset_inputs, dataset_outputs))
dataset = dataset.shuffle(100_000, seed=42, reshuffle_each_iteration=False)

cut = int(df_ratings.shape[0] * 0.8)
train_dataset = dataset.take(cut).batch(1024).cache()
test_dataset = dataset.skip(df_ratings.shape[0] - cut).batch(1024).cache()

## Modelo de retrieval

### Claude explanation (not from the professor)
**Two-tower retrieval model.** Each tower is just an embedding, one for readers (the query) and one for books (the candidates). Their dot product is the score.
- `build` makes the retrieval layer reuse the book embedding table as its list of candidates.
- `call` returns the reader's embedding and, outside training, the top-`k` books found by `BruteForceRetrieval` (dot product against every book).
- `compute_loss` trains the dot product to match the rescaled rating, using MSE.

In [6]:
class RetrievalModel(keras.Model):
    """Modelo de recuperación.

    Args:
      num_users: Número de usuarios para el embedding.
      num_candidates: Número de candidatos para el embedding..
      embedding_dimension: Dimensión del embedding.
    """

    def __init__(
        self,
        num_users,
        num_candidates,
        embedding_dimension=256,
        k=10,
        **kwargs,
    ):
        super().__init__(**kwargs)

        # Torre de consulta (users), sólo un embedding
        self.user_embedding = keras.layers.Embedding(num_users, embedding_dimension)

        # Torre de candidatos (items), sólo un embedding
        self.candidate_embedding = keras.layers.Embedding(num_candidates, embedding_dimension)

        # La capa que hace la recuperación
        self.retrieval = keras_rs.layers.BruteForceRetrieval(k=k, return_scores=True)

        self.loss_fn = keras.losses.MeanSquaredError()

    def build(self, input_shape):
        self.user_embedding.build(input_shape)
        self.candidate_embedding.build(input_shape)
        # En este caso, los candidatos son directamente los embeddings del item.
        # Reusamos la variable para ahorrarnos pasos.
        self.retrieval.candidate_embeddings = self.candidate_embedding.embeddings
        self.retrieval.build(input_shape)
        super().build(input_shape)

    def call(self, inputs, training=False):
        user_embeddings = self.user_embedding(inputs)
        result = { "user_embeddings": user_embeddings }

        if not training:
            # Durante el entrenamiento no necesito calcular predicciones
            result["predictions"] = self.retrieval(user_embeddings)
        return result

    def compute_loss(self, x, y, y_pred, sample_weight, training=True):
        candidate_id = y["item_id"]
        rating = y["rating"]
        user_embeddings = y_pred["user_embeddings"]
        candidate_embeddings = self.candidate_embedding(candidate_id)

        labels = keras.ops.expand_dims(rating, -1)

        # Calcula el score de afinidad multiplicando los dos embeddings
        scores = keras.ops.sum(
            keras.ops.multiply(user_embeddings, candidate_embeddings),
            axis=1,
            keepdims=True,
        )
        return self.loss_fn(labels, scores, sample_weight)

## Entrenamiento y evaluación

### Claude explanation (not from the professor)
Creates the model to return the top 100 books and compiles it. No loss is passed because the model defines its own `compute_loss`.

In [8]:
retrieval_model = RetrievalModel(len(unique_user_id) + 1, len(unique_item_id) + 1, k=100)
retrieval_model.compile(optimizer=keras.optimizers.Adagrad(learning_rate=0.1))

### Claude explanation (not from the professor)
Trains for 10 epochs and evaluates on the test set every 5. `batch_size` is ignored because the datasets are already batched.

In [9]:
history = retrieval_model.fit(train_dataset, validation_data=test_dataset, validation_freq=5, epochs=10, batch_size=256)

/Users/sergiovolta/repos/sr_uba_2026/scraping/.venv/lib/python3.13/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


Epoch 1/10
  1/361 ━━━━━━━━━━━━━━━━━━━━ 1:01 171ms/step - loss: 0.4224

W0000 00:00:1792250380.075318 2854058 cache_dataset_ops.cc:912] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.


361/361 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - loss: 0.4220
Epoch 2/10
361/361 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - loss: 0.4217
Epoch 3/10
361/361 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - loss: 0.4215
Epoch 4/10
361/361 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - loss: 0.4213
Epoch 5/10
361/361 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - loss: 0.4211 - val_loss: 0.4273
Epoch 6/10
361/361 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - loss: 0.4208
Epoch 7/10
361/361 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - loss: 0.4206
Epoch 8/10
361/361 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - loss: 0.4204
Epoch 9/10
361/361 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - loss: 0.4201
Epoch 10/10
361/361 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - loss: 0.4199 - val_loss: 0.4264


## Predicción

### Claude explanation (not from the professor)
Retrieves the top 100 books for `popocito`. `predictions` holds `(scores, indices)`, so `[1][0]` are the book indices.

**Note:** the loop prints each book's integer index instead of its `score`. Retrieval also doesn't exclude books the reader already rated.

In [10]:
id_lector = user_id_2_id_mapping["popocito"]
predictions = retrieval_model.predict(keras.ops.convert_to_tensor([id_lector]))
predictions = keras.ops.convert_to_numpy(predictions["predictions"])

print(f"Libros recomendados para el lector {id_lector}:")
for item_id, score in zip(predictions[1][0], predictions[0][0]):
    print(id_2_item_id_mapping[item_id], "-->", item_id)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
Libros recomendados para el lector 1322:
el-carro-de-hierro --> 36735.0
seis-peces-azules --> 46482.0
la-aurora-de-los-elegidos --> 30198.0
la-biblia-de-los-caidos-tomo-1-del-testamento-de-sombra --> 27439.0
la-joya-de-medina --> 15898.0
freud-en-las-pampas --> 36807.0
la-ciudadela-de-las-piedras --> 18966.0
el-olvido-de-la-razon --> 32122.0
dama-de-treboles --> 20801.0
winter --> 35514.0
desgracia --> 4841.0
hal-foster-principe-de-ilustradores --> 40433.0
la-leyenda-de-sleepy-hollow --> 3163.0
mis-documentos --> 8490.0
andalucia-durante-la-guerra-de-la-independencia --> 28994.0
album-de-boda --> 11559.0
la-candidata --> 21320.0
el-taller-de-munecas --> 41488.0
the-raven-boys-los-saqueadores-de-suenos --> 47040.0
todavia-1 --> 43867.0
los-nuevos-curas --> 32774.0
destinada-a-sentir --> 16732.0
el-paraiso --> 17667.0
la-mecanica-del-corazon --> 1542.0
relatos-de-terror-1 --> 18497.0
las-cartas-del-boom --> 28798.0
sobre-la-historia-natural-de-la-des

# Modelo de Ranking

### Claude explanation (not from the professor)
Second stage: the **ranking** dataset. Inputs are the reader and book indices, and the target is the rating rescaled to 0–1 (÷ 9). **Note:** `skip(df_ratings.shape[0] - cut)` skips only 20% of the rows, so the test set is the *last 80%* and mostly overlaps training. It should be `skip(cut)`.

In [11]:
dataset_inputs = {
    "user_id": df_ratings['id_lector'].to_numpy(dtype="float32"),
    'item_id': df_ratings['id_libro'].to_numpy(dtype="float32")
}

dataset_outputs = (df_ratings['rating'].to_numpy() - 1.0) / 9.0

dataset = tf.data.Dataset.from_tensor_slices((dataset_inputs, dataset_outputs))
dataset = dataset.shuffle(100_000, seed=42, reshuffle_each_iteration=False)

cut = int(df_ratings.shape[0] * 0.8)
train_dataset = dataset.take(cut).batch(256).cache()
test_dataset = dataset.skip(df_ratings.shape[0] - cut).batch(256).cache()

## Modelo

### Claude explanation (not from the professor)
**Ranking model**: reader and book embeddings (32 dimensions) concatenated and fed to dense layers that predict the rescaled rating. It's more expressive than a dot product but too slow to run against every book, which is why retrieval narrows the candidates first.

In [12]:
class RankingModel(keras.Model):
    """Crea un modelo de ranking

    Args:
      num_users: Number of entries in the user embedding table.
      num_items: Number of entries in the candidate embedding table.
      embedding_dimension: Output dimension for user and movie embedding tables.
    """

    def __init__(
        self,
        num_users,
        num_items,
        embedding_dimension=32,
        **kwargs,
    ):
        super().__init__(**kwargs)
        # Embedding de los usuarios
        self.user_embedding = keras.layers.Embedding(num_users, embedding_dimension)

        # Embedding de los items
        self.item_embedding = keras.layers.Embedding(num_items, embedding_dimension)

        # Predicciones
        self.ratings = keras.Sequential(
            [
                # Capas densas
                keras.layers.Dense(256, activation="relu"),
                keras.layers.Dense(64, activation="relu"),

                # Predicciones en la capa final
                keras.layers.Dense(1, activation="sigmoid"),
            ]
        )

    def call(self, inputs):
        user_id, movie_id = inputs["user_id"], inputs["item_id"]
        user_embeddings = self.user_embedding(user_id)
        item_embeddings = self.item_embedding(movie_id)
        return self.ratings(keras.ops.concatenate([user_embeddings, item_embeddings], axis=1))

## Entrenamiento y evaluación

### Claude explanation (not from the professor)
Creates the model (`+1` leaves room for one extra index) and compiles it with MSE loss, RMSE as the metric and the Adagrad optimizer.

In [13]:
ranking_model = RankingModel(len(unique_user_id) + 1, len(unique_item_id) + 1)
ranking_model.compile(
    loss=keras.losses.MeanSquaredError(),
    metrics=[keras.metrics.RootMeanSquaredError()],
    optimizer=keras.optimizers.Adagrad(learning_rate=0.1),
)

### Claude explanation (not from the professor)
Trains for 10 epochs and evaluates on the test set every 5. `batch_size` is ignored because the datasets are already batched.

In [14]:
history = ranking_model.fit(train_dataset, validation_data=test_dataset, validation_freq=5, epochs=10, batch_size=512)

/Users/sergiovolta/repos/sr_uba_2026/scraping/.venv/lib/python3.13/site-packages/jax/_src/core.py:1976: UserWarning: Explicitly requested dtype float64 requested in ShapeDtypeStruct is not available, and will be truncated to dtype float32. To enable more dtypes, set the jax_enable_x64 configuration option or the JAX_ENABLE_X64 shell environment variable. See https://github.com/jax-ml/jax#current-gotchas for more.
  return aval_fn(x)
W0000 00:00:1792281140.374808 2854058 cache_dataset_ops.cc:912] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.


Epoch 1/10
1442/1442 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - loss: 0.0401 - root_mean_squared_error: 0.2002 
Epoch 2/10
1442/1442 ━━━━━━━━━━━━━━━━━━━━ 1s 896us/step - loss: 0.0393 - root_mean_squared_error: 0.1984
Epoch 3/10
1442/1442 ━━━━━━━━━━━━━━━━━━━━ 1s 855us/step - loss: 0.0372 - root_mean_squared_error: 0.1928
Epoch 4/10
1442/1442 ━━━━━━━━━━━━━━━━━━━━ 1s 856us/step - loss: 0.0344 - root_mean_squared_error: 0.1854
Epoch 5/10
1442/1442 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.0328 - root_mean_squared_error: 0.1810 - val_loss: 0.0339 - val_root_mean_squared_error: 0.1840
Epoch 6/10
1442/1442 ━━━━━━━━━━━━━━━━━━━━ 1s 844us/step - loss: 0.0318 - root_mean_squared_error: 0.1782
Epoch 7/10
1442/1442 ━━━━━━━━━━━━━━━━━━━━ 1s 826us/step - loss: 0.0311 - root_mean_squared_error: 0.1762
Epoch 8/10
1442/1442 ━━━━━━━━━━━━━━━━━━━━ 1s 858us/step - loss: 0.0305 - root_mean_squared_error: 0.1748
Epoch 9/10
1442/1442 ━━━━━━━━━━━━━━━━━━━━ 1s 868us/step - loss: 0.0301 - root_mean_squared_error: 0.1736
E

## Predicción

### Claude explanation (not from the professor)
The full **retrieve-then-rank** pipeline for `popocito`:
1. The retrieval model quickly picks 100 candidates.
2. The ranking model predicts a rating for each candidate.
3. Predictions are mapped back to 1–10.

**Note:** the retrieval call uses `id_lector` from an earlier cell, not `user_id`. Both hold the same value here, but changing only one would break it. Results print in retrieval order; sort by `prediction` to get the final ranking.

In [15]:
user_id = user_id_2_id_mapping["popocito"]

predictions = retrieval_model.predict(keras.ops.convert_to_tensor([id_lector]))
predictions = keras.ops.convert_to_numpy(predictions["predictions"])

item_ids = predictions[1][0]

predictions = ranking_model.predict({
    "user_id": keras.ops.array([user_id] * len(item_ids)),
    "item_id": keras.ops.array(item_ids),
})

predictions = keras.ops.convert_to_numpy(keras.ops.squeeze(predictions, axis=1))

for id_libro, prediction in zip(item_ids, predictions):
    print(f"{id_2_item_id_mapping[id_libro]} --> {prediction *9.0 + 1.0:,.2f}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step
el-carro-de-hierro --> 7.27
seis-peces-azules --> 6.27
la-aurora-de-los-elegidos --> 6.25
la-biblia-de-los-caidos-tomo-1-del-testamento-de-sombra --> 6.60
la-joya-de-medina --> 6.27
freud-en-las-pampas --> 6.57
la-ciudadela-de-las-piedras --> 6.29
el-olvido-de-la-razon --> 6.55
dama-de-treboles --> 6.57
winter --> 6.66
desgracia --> 7.09
hal-foster-principe-de-ilustradores --> 5.89
la-leyenda-de-sleepy-hollow --> 6.11
mis-documentos --> 6.24
andalucia-durante-la-guerra-de-la-independencia --> 6.00
album-de-boda --> 7.06
la-candidata --> 7.20
el-taller-de-munecas --> 6.23
the-raven-boys-los-saqueadores-de-suenos --> 6.75
todavia-1 --> 6.41
los-nuevos-curas --> 6.61
destinada-a-sentir --> 6.44
el-paraiso --> 7.01
la-mecanica-del-corazon --> 5.54
relatos-de-terror-1 --> 6.73
las-cartas-del-boom --> 6.07
sobre-la-historia-natural-de-la-destruccion --> 6.61
el-principe-maldito --> 6.18
el-mus --> 6.47
cartas-a-palaci